In [ ]:
#Import libraries 
import sys
from pathlib import Path
import re
import json
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import fasttext

from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


sys.path.append(str(Path.cwd().parent / "src"))
from helpers import pick_by_pos, write_fasttext, train_fasttext, train_svm, eval_metrics_ft, normalize_doc_id, undersample_quantitative, ramp, select_for_iteration_svm, select_for_iteration

#Set random seeds to ensure reproducibility of results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

#Experiment parameters
chunk_size = 2000
max_iterations = 60
target_qual_auto = 4000
test_per_class = 20

# Dynamic confidence schedule
TAU_START = 0.70
TAU_END = 0.92
ramp_iteration = max_iterations

# Class imbalance controls
max_quant_to_qual_ratio= 1.5
max_quant_add_per_iteration= 200
undersample_quant_to_qual = 1.5

# Heuristic candidate review sample
heuristic_sample_frac = 0.03

# Two experiment configurations 
fastText_configuration = [
      {
        "name": "cfg_A",
        "epoch": 25,
        "lr": 0.5,
        "wordNgrams": 1,
        "dim": 100,
        "loss": "softmax",
        "minCount": 1,
        "minCountLabel": 0,
        "ws": 5,
        "thread": 4,
    },
    {
        "name": "cfg_B",
        "epoch": 35,
        "lr": 0.7,
        "wordNgrams": 2,
        "dim": 100,
        "loss": "softmax",
        "minCount": 2,
        "minCountLabel": 0,
        "ws": 5,
        "thread": 4,
    },
]

best_metric = "f1_macro"
base_dir = Path.cwd().parent
data_dir = base_dir / "data"
output_dir = base_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
base_train_pool = pd.read_csv(output_dir / "base_train_pool.csv")
eval_df = pd.read_csv(output_dir / "test_df_stage0.csv")
cs_df = pd.read_csv(data_dir / "data.csv")

In [16]:

label_qual = "qualitative"
label_quant = "quantitative"

##Store per-iteration results 
ft_experiment_logs = []
ft_model_registry = []

# Loop through predifined FastText configuration 
for cfg in fastText_configuration:
    cfg_name = cfg["name"]
    print(f"\n=== Running {cfg_name} ===")

   ## Initial Training Pool Setup 
    train_pool = base_train_pool[["doc_id", "title", "abstract", "label", "label_norm"]].copy()
    train_pool = normalize_doc_id(train_pool)

  ##Initial Model Training
    train_fit = undersample_quantitative(
        train_pool[["doc_id", "title", "abstract", "label"]],
        qual_label=label_qual,
        quant_label=label_quant,
        ratio=undersample_quant_to_qual,
        random_state=RANDOM_SEED,
    )

    iter0_txt = output_dir / f"{cfg_name}_train_iter_0.txt"
    iter0_model = output_dir / f"{cfg_name}_model_iter_0.ftz"
    print("output_dir:", output_dir.resolve())
    print("iter0_txt:", iter0_txt.resolve())
    
    write_fasttext(train_fit, "abstract", "label", iter0_txt)
    print("txt exists after write:", iter0_txt.exists())
    
    model = train_fasttext(iter0_txt, iter0_model, cfg)
    current_model_path = iter0_model

    ## Evaluation 
    m0 = eval_metrics_ft(model, eval_df, y_col="y_true")
    ft_experiment_logs.append({
        "config": cfg_name,
        "iter": 0,
        "tau": np.nan,
        "chunk_size": 0,
        "auto_total_iter": 0,
        "manual_total_iter": 0,
        "qual_auto_iter": 0,
        "quant_auto_iter": 0,
        "added_unique": 0,
        "added_qual_unique": 0,
        "added_quant_unique": 0,
        "train_pool_size": int(len(train_pool)),
        "fit_train_size": int(len(train_fit)),
        "added_qual_cum": 0,
        "acc": round(m0["acc"], 4) if pd.notna(m0["acc"]) else np.nan,
        "f1_macro": round(m0["f1_macro"], 4) if pd.notna(m0["f1_macro"]) else np.nan,
        "f1_qual": round(m0["f1_qual"], 4) if pd.notna(m0["f1_qual"]) else np.nan,
        "f1_quant": round(m0["f1_quant"], 4) if pd.notna(m0["f1_quant"]) else np.nan,
        "precision_macro": round(m0["precision_macro"], 4) if pd.notna(m0["precision_macro"]) else np.nan,
        "recall_macro": round(m0["recall_macro"], 4) if pd.notna(m0["recall_macro"]) else np.nan,
        "model_path": str(iter0_model.resolve()),
    })
    ft_model_registry.append({
        "config": cfg_name,
        "iter": 0,
        "model_path": str(iter0_model.resolve()),
        "fit_train_size": int(len(train_fit)),
        "train_pool_size": int(len(train_pool)),
        "acc": m0["acc"],
        "f1_macro": m0["f1_macro"],
    })

    # Build Unlabelled Pool
    seen_ids = set(train_pool["doc_id"].astype(str))
    unlabeled_pool = cs_df.copy()
    unlabeled_pool["doc_id"] = unlabeled_pool["doc_id"].astype(str).str.strip()
    unlabeled_pool = unlabeled_pool[~unlabeled_pool["doc_id"].isin(seen_ids)].copy().reset_index(drop=True)

    added_qual_cum = 0

    ###############################################
    # Iterative Training Loop 
    ###############################################

    for it in range(1, max_iterations + 1):
        if unlabeled_pool.empty:
            break

        ##Tresholding ramping 
        tau_it = ramp(it, TAU_START, TAU_END, ramp_iteration)

        ##Select Chunk of Unlabelled Data
        chunk = unlabeled_pool.iloc[:chunk_size].copy()
        unlabeled_pool = unlabeled_pool.iloc[chunk_size:].copy()

        ## Select predictions for auto labelling or manual labelling 
        auto, manual = select_for_iteration(model, chunk, tau=tau_it)

        for df_ in (auto, manual):
            df_["pred_norm"] = (
                df_["pred_norm"].astype(str)
                .str.replace("__label__", "", regex=False)
                .str.strip().str.lower()
            )

        #Split auto labelled by class
        qual_auto = auto[auto["pred_norm"] == label_qual].copy()
        quant_auto = auto[auto["pred_norm"] == label_quant].copy()

        ##Build new training row
        new_rows = pd.concat([
            qual_auto.assign(label=label_qual)[["doc_id", "title", "abstract", "label"]],
            quant_auto.assign(label=label_quant)[["doc_id", "title", "abstract", "label"]],
        ], ignore_index=True)
        new_rows = normalize_doc_id(new_rows)

        ## Remove duplicates
        existing_ids = set(train_pool["doc_id"].astype(str))
        truly_new = new_rows[~new_rows["doc_id"].astype(str).isin(existing_ids)].copy()

        truly_new["label_norm"] = truly_new["label"].astype(str).str.strip().str.lower()
        truly_new_qual = truly_new[truly_new["label_norm"] == label_qual].copy()
        truly_new_quant = truly_new[truly_new["label_norm"] == label_quant].copy()

        ## Control Class Balance
        cur_qual = int((train_pool["label_norm"] == label_qual).sum())
        cur_quant = int((train_pool["label_norm"] == label_quant).sum())

        projected_qual = cur_qual + len(truly_new_qual)
        quant_allowed_by_ratio = int(max_quant_to_qual_ratio * max(projected_qual, 1)) - cur_quant
        quant_allowed = max(0, min(quant_allowed_by_ratio, max_quant_add_per_iteration))

        ##Limit Quantitative Addition
        if len(truly_new_quant) > quant_allowed:
            conf_map = auto.set_index("doc_id")["confidence"] if "confidence" in auto.columns else None
            if conf_map is not None:
                truly_new_quant["confidence"] = truly_new_quant["doc_id"].map(conf_map)
                truly_new_quant = truly_new_quant.sort_values("confidence", ascending=False).head(quant_allowed).copy()
            else:
                truly_new_quant = truly_new_quant.head(quant_allowed).copy()

        #New training data
        final_new = pd.concat([truly_new_qual, truly_new_quant], ignore_index=True)
        added = len(final_new)
        added_qual = len(truly_new_qual)
        added_quant = len(truly_new_quant)

        iter_model = output_dir / f"{cfg_name}_model_iter_{it}.ftz"

        #Update training pool and retrain model 
        if added > 0:
            train_pool = pd.concat([train_pool, final_new], ignore_index=True)
            train_pool = train_pool.drop_duplicates(subset="doc_id", keep="first").copy()
            train_pool["label_norm"] = train_pool["label"].astype(str).str.strip().str.lower()

            train_fit = undersample_quantitative(
                train_pool[["doc_id", "title", "abstract", "label"]],
                qual_label=label_qual,
                quant_label=label_quant,
                ratio=undersample_quant_to_qual,
                random_state=RANDOM_SEED,
            )
            fit_train_size = len(train_fit)

            iter_txt = output_dir / f"{cfg_name}_train_iter_{it}.txt"
            print(iter_txt)
            write_fasttext(train_fit, "abstract", "label", iter_txt)
            model = train_fasttext(iter_txt, iter_model, cfg)
            current_model_path = iter_model
        else:
            fit_train_size = np.nan
            shutil.copy2(current_model_path, iter_model)
            current_model_path = iter_model
   
        ##Evaluate updated model
        m = eval_metrics_ft(model, eval_df, y_col="y_true")
        added_qual_cum += added_qual

        #Log results
        ft_experiment_logs.append({
            "config": cfg_name,
            "iter": it,
            "tau": round(float(tau_it), 4),
            "chunk_size": int(len(chunk)),
            "auto_total_iter": int(len(auto)),
            "manual_total_iter": int(len(manual)),
            "qual_auto_iter": int(len(qual_auto)),
            "quant_auto_iter": int(len(quant_auto)),
            "added_unique": int(added),
            "added_qual_unique": int(added_qual),
            "added_quant_unique": int(added_quant),
            "train_pool_size": int(len(train_pool)),
            "fit_train_size": int(fit_train_size) if pd.notna(fit_train_size) else np.nan,
            "added_qual_cum": int(added_qual_cum),
            "acc": round(m["acc"], 4) if pd.notna(m["acc"]) else np.nan,
            "f1_macro": round(m["f1_macro"], 4) if pd.notna(m["f1_macro"]) else np.nan,
            "f1_qual": round(m["f1_qual"], 4) if pd.notna(m["f1_qual"]) else np.nan,
            "f1_quant": round(m["f1_quant"], 4) if pd.notna(m["f1_quant"]) else np.nan,
            "precision_macro": round(m["precision_macro"], 4) if pd.notna(m["precision_macro"]) else np.nan,
            "recall_macro": round(m["recall_macro"], 4) if pd.notna(m["recall_macro"]) else np.nan,
            "model_path": str(iter_model.resolve()),
        })

        ft_model_registry.append({
            "config": cfg_name,
            "iter": it,
            "model_path": str(iter_model.resolve()),
            "fit_train_size": int(fit_train_size) if pd.notna(fit_train_size) else np.nan,
            "train_pool_size": int(len(train_pool)),
            "acc": m["acc"],
            "f1_macro": m["f1_macro"],
        })

        ##Print Progress
        print(
            f"{cfg_name} it={it} tau={tau_it:.3f} chunk={len(chunk)} "
            f"auto={len(auto)} added={added} (+q={added_qual},+n={added_quant}) "
            f"f1m={m['f1_macro']:.4f}"
        )

        if added_qual_cum >= target_qual_auto:
            break

log_df = pd.DataFrame(ft_experiment_logs)
ft_model_registry_df = pd.DataFrame(ft_model_registry)

log_df.to_csv(output_dir/ "ft_experiment_logs.csv", index=False)
ft_model_registry_df.to_csv(output_dir/ "ft_model_registry.csv", index=False)

display(log_df.head())
print("Saved:", output_dir / "ft_experiment_logs.csv")
print("Saved:", output_dir/ "ft_model_registry.csv")
 


=== Running cfg_A ===
output_dir: /Users/brandidennis/Desktop/ClassiDocs/NLP Pipeline/output
iter0_txt: /Users/brandidennis/Desktop/ClassiDocs/NLP Pipeline/output/cfg_A_train_iter_0.txt
txt exists after write: True


Read 0M words
Number of words:  12269
Number of labels: 2
Progress: 100.0% words/sec/thread: 3872741 lr:  0.000000 avg.loss:  0.212474 ETA:   0h 0m 0s


/Users/brandidennis/Desktop/ClassiDocs/NLP Pipeline/output/cfg_A_train_iter_1.txt


Read 0M words
Number of words:  24788
Number of labels: 2
Progress: 100.0% words/sec/thread: 5836291 lr:  0.000000 avg.loss:  0.094311 ETA:   0h 0m 0s
Read 0M words
Number of words:  34017
Number of labels: 2


cfg_A it=1 tau=0.700 chunk=2000 auto=1647 added=303 (+q=303,+n=0) f1m=0.7954
/Users/brandidennis/Desktop/ClassiDocs/NLP Pipeline/output/cfg_A_train_iter_2.txt


...

Read 1M words
Number of words:  31246
Number of labels: 2
Progress:  97.8% words/sec/thread: 3057024 lr:  0.015085 avg.loss:  0.016389 ETA:   0h 0m 0s

cfg_B it=10 tau=0.734 chunk=2000 auto=1827 added=791 (+q=591,+n=200) f1m=0.7248


Progress: 100.0% words/sec/thread: 3026504 lr:  0.000000 avg.loss:  0.016073 ETA:   0h 0m 0s


,config,iter,tau,chunk_size,auto_total_iter,manual_total_iter,qual_auto_iter,quant_auto_iter,added_unique,added_qual_unique,...,train_pool_size,fit_train_size,added_qual_cum,acc,f1_macro,f1_qual,f1_quant,precision_macro,recall_macro,model_path
0,cfg_A,0,NaN,0,0,0,0,0,0,0,...,1155,375,0,0.775,0.7714,0.7429,0.8000,0.7933,0.775,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
1,cfg_A,1,0.7000,2000,1647,353,303,1344,303,303,...,1458,1132,303,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
2,cfg_A,2,0.7037,2000,1842,158,300,1542,424,300,...,1882,1882,603,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
3,cfg_A,3,0.7075,2000,1882,118,319,1563,519,319,...,2401,2401,922,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
4,cfg_A,4,0.7112,2000,1898,102,367,1531,567,367,...,2968,2968,1289,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...


Saved: /Users/brandidennis/Desktop/ClassiDocs/NLP Pipeline/output/ft_experiment_logs.csv
Saved: /Users/brandidennis/Desktop/ClassiDocs/NLP Pipeline/output/ft_model_registry.csv


In [17]:
log_df

,config,iter,tau,chunk_size,auto_total_iter,manual_total_iter,qual_auto_iter,quant_auto_iter,added_unique,added_qual_unique,...,train_pool_size,fit_train_size,added_qual_cum,acc,f1_macro,f1_qual,f1_quant,precision_macro,recall_macro,model_path
0,cfg_A,0,NaN,0,0,0,0,0,0,0,...,1155,375,0,0.775,0.7714,0.7429,0.8000,0.7933,0.775,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
1,cfg_A,1,0.7000,2000,1647,353,303,1344,303,303,...,1458,1132,303,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
2,cfg_A,2,0.7037,2000,1842,158,300,1542,424,300,...,1882,1882,603,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
3,cfg_A,3,0.7075,2000,1882,118,319,1563,519,319,...,2401,2401,922,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
4,cfg_A,4,0.7112,2000,1898,102,367,1531,567,367,...,2968,2968,1289,0.800,0.7954,0.7647,0.8261,0.8297,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
5,cfg_A,5,0.7149,2000,1872,128,411,1461,611,411,...,3579,3579,1700,0.825,0.8222,0.8000,0.8444,0.8467,0.825,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
6,cfg_A,6,0.7186,2000,1870,130,471,1399,671,471,...,4250,4250,2171,0.800,0.7980,0.7778,0.8182,0.8125,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
7,cfg_A,7,0.7224,2000,1874,126,478,1396,678,478,...,4928,4928,2649,0.800,0.7980,0.7778,0.8182,0.8125,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
8,cfg_A,8,0.7261,2000,1886,114,539,1347,739,539,...,5667,5667,3188,0.800,0.7980,0.7778,0.8182,0.8125,0.800,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
9,cfg_A,9,0.7298,2000,1873,127,604,1269,804,604,...,6471,6471,3792,0.775,0.7737,0.7568,0.7907,0.7813,0.775,/Users/brandidennis/Desktop/ClassiDocs/NLP Pip...
